TODO :
- Decide what one row represents and write it down (for example: a repository, a film, a weather station, or an article). As Exercises 6 and 7 showed, this choice determines what your clusters mean.

    - A row is a game or stream. Our features are viewer counts, tag one-hot, hour-of-day profile

- Build the feature table X. The table below suggests features for each project source. Use one-hot encoding for categories, take the log of skewed counts, and use TF-IDF for text columns.

- Standardise the features ((feat - feat.mean()) / feat.std()), then cluster for two or three values of 𝑘. If you compute silhouette scores on more than a few thousand rows, pass sample_size=10_000 to silhouette_score so that the 𝑂(𝑛2) pairwise distance calculation does not hang.

- Print the cluster profile table and give every cluster a short descriptive name. Put the cluster names and interpretation in your report, and the full profile table in the appendix.

- Inspect any small or hard-to-name cluster. As with the 14 broken-meter trips in Exercise 6, an unusual cluster often reveals a data-quality issue missed in the previous weeks.

In [48]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.width", 100)
sns.set_theme(style="whitegrid")

CLEAN_DATA_DIR = Path.cwd().parent / "data/clean/"

clean_games_path = CLEAN_DATA_DIR / "games_cleaned.parquet"
clean_clips_path = CLEAN_DATA_DIR / "clips_cleaned.parquet"
clean_channels_path = CLEAN_DATA_DIR / "channels_cleaned.parquet"

cgdf = pd.read_parquet(clean_games_path)
cdf = pd.read_parquet(clean_clips_path)
chdf = pd.read_parquet(clean_channels_path)

### Feature selection

First we decide what one row represents. In our case we have 3 different data frames, we can join them together to get a very rich representation of each clip.

Features we will include : game_name, clip_language, clip_title, view_count, duration, is_featured, time posted, is branded, content classification labels, tags, content_classification_labels

## Building the feature table

### Content Classification
Here, we use one-hot encoding for the different labels present in content classification column.

In [49]:
# One-hot encoding and bag of words
# one-hot encode the content_classification_labels column
import re

# extract non-empty labels
label_set = {
    label
    for text in chdf['content_classification_labels'].dropna()
    for label in re.findall(r"'([^']+)'", str(text))
    if label.strip() not in ("[]", "")
}

for label in label_set:
    chdf[f"label_{label}"] = (
        chdf["content_classification_labels"]
        .astype("string")
        .str.contains(f"'{label}'", regex=False, na=False)
        .astype("int8")
    )

# label_none = 1 if all category columns sum to 0
label_cols = [f"label_{label}" for label in label_set]
chdf["label_none"] = (chdf[label_cols].sum(axis=1) == 0).astype("int8")

chdf = chdf.drop(columns=["content_classification_labels"])

print(chdf[:5])

   broadcaster_id broadcaster_language  game_id          game_name  \
0       234973404                   en   509658      Just Chatting   
1       683970274                   en   509658      Just Chatting   
2       552120296                   en    18122  World of Warcraft   
3       423486275                   ru   509658      Just Chatting   
4           12826                   en   509658      Just Chatting   

                                               title  \
0  IRL Shanghai 🇨🇳 I'm back with @pokimane 🥰 Dres...   
1  SOUL FOOD COOKING - FRIED CHICKEN, MAC N CHEES...   
2  [DROPS ON] UNBANNED WOW FOREVER BETA FINAL DAY...   
3  ПОДГОТОВКА К ГРЯЗНОЙ ФОТОСЕССИИ С ДЕВОЧКАМИ)) ...   
4  Twitch Public Access (October 2026) | Celebrat...   

                                                tags  is_branded_content  label_SexualThemes  \
0  [['Travel' 'food' 'lifestyle' 'Chinese中文' 'IRL...               False                   0   
1  [['american' 'girl' 'asian' 'canadian' 'Englis.

### Tags

Keep in mind that the most represented tags are language tags. We could try to remove the language tags to fix this issue. 

The probelem is that if we add 400 tags, the clustering will be overwhelmingly driven by the tags and not the other columns. We need to make a choice about which tags we will represent. It could make sense to include only the most popular tags. 

Another issue remains : tags are often language based: a german game has german language tags , same for russian, english etc. The tag for 'win' will be 'ganar' in spanish channels, 'vincere' in italian, and so on, with many different syntaxes for the same concept. This is bad because we already have a language variable, we want the tag concept as a source of new information.

This opens up a lot of other problems, namely that non-English tags are mutually exclusive across language communities, the clusters will naturally group rows by the language or even alphabet of their tags, instead of the underlying concepts that they represent.

In [50]:
import re

# extract tags from list
def parse_tags(val):
    if val is None or pd.isna(val):
        return []
    return [t.strip().lower() for t in re.findall(r"'([^']+)'", str(val)) if t.strip()]

chdf["clean_tags"] = chdf["tags"].apply(parse_tags)

# language tags identified
lang_tags_to_remove = {
    "english", "русский", "español", "日本語", "deutsch", "français",
    "português", "italiano", "polski", "türkçe", "한국어", "chinese中文",
    "中文", "ukr", "українська", "عربي", "čeština", "magyar", "suomi", "العربية", "arabic", "nederlands"
}

# remove language tags
chdf["concept_tags"] = chdf["clean_tags"].apply(
    lambda tags: [t for t in tags if t not in lang_tags_to_remove]
)

# visualize top tags
exploded_concepts = chdf["concept_tags"].explode()
print(f"Total unique non-language tags: {exploded_concepts.nunique()}")
print("\nTop 50 Concept Tags:")
print(exploded_concepts.value_counts().head(50))

Total unique non-language tags: 1536

Top 50 Concept Tags:
concept_tags
dropsenabled       135
drops有効             34
vtuber              33
dropsвключены       31
irl                 28
adhd                24
fps                 23
funny               21
启用掉宝                19
gaming              19
chill               19
ストグラ                18
anime               16
esports             16
pvp                 16
girl                16
warzone             15
ranked              15
dropsactivados      15
deadlock            14
drops               13
arcraiders          13
feet                13
overwatch           12
slots               12
fortnite            12
osrs                12
gta                 11
nopixelv            11
nopixel             11
雑談                  10
cs2                 10
egirl               10
competitive         10
calldofduty          8
cod                  8
minecraft            8
dropsactivés         8
valorant             8
fc27                 8
educatio

To work around the problems with the tags outlined above, we decided to create macro tags buckets that represent the content of a channel.  

To do this, we manually went over the unique-non language tags and built a dictionary to map common tags to 6 different content buckets:
- drops
- competitive
- vtuber/anime
- irl/chill
- multiplayer roleplay
- persona

These were the most common content types we identified. Of course, there is a subjective element to this. The idea is to assign the most popular concepts to a content type, to have a good chance at categorizing lots of channels meaningfully, independent of language, while still being able to cluster on features other than just tags.

In [51]:

SUPER_TAGS = {
    "tag_drops": ["drop", "掉宝", "дроп"],
    "tag_competitive": ["rank", "comp", "esport", "tourn", "pvp", "fps", "tryhard", "eternus"],
    "tag_vtuber_anime": ["vtub", "anime", "png", "weeb", "otaku", "manga"],
    "tag_irl_chill": ["irl", "chill", "chat", "asmr", "vibe", "travel", "food", "music", "雑談", "funny", "cozy", "lifestyle", "cooking"],
    "tag_rp_multiplayer": ["nopixel", "roleplay", "gtarp", "gta5rp", "multiplayer", "coop", "ストグラ", "squad"],
    "tag_persona": ["girl", "adhd", "lgbt", "feet", "bald", "educational"]
}

# One-hot encode super-tag
for col_name, keywords in SUPER_TAGS.items():
    chdf[col_name] = chdf["concept_tags"].apply(
        lambda tag_list: int(any(kw in tag for tag in tag_list for kw in keywords))
    )

# Check how many channels matched each super-tag
print(chdf[list(SUPER_TAGS.keys())].sum())

chdf["num_concept_tags"] = chdf["concept_tags"].apply(len)

tag_drops             274
tag_competitive        93
tag_vtuber_anime       44
tag_irl_chill         130
tag_rp_multiplayer     43
tag_persona            78
dtype: int64


### Merge clips, channels, games into one single feature df X

In [52]:
# Map game_name directly from cgdf using the clip's actual game_id
game_map = dict(zip(cgdf["id"], cgdf["name"]))
cdf["game_name"] = cdf["game_id"].map(game_map)

# Drop duplicate columns from chdf before merging
chdf_cols_to_keep = (
    ["broadcaster_id", "is_branded_content"]
    + label_cols + ["label_none"]
    + list(SUPER_TAGS.keys())
)

df = pd.merge(cdf, chdf[chdf_cols_to_keep], on="broadcaster_id", how="left")

# Build num features
X = pd.DataFrame(index=df.index)
X["log_view_count"] = np.log1p(df["view_count"].astype(float))
X["duration"] = df["duration"].astype(float)
X["is_featured"] = df["is_featured"].fillna(False).astype(int)
X["is_branded_content"] = df["is_branded_content"].fillna(False).astype(int)

# One-hot encode day of week
day_dummies = pd.get_dummies(df["created_at"].dt.day_name(), prefix="day", dtype=int)

# Top 6 languages + 'other'
top_langs = ["en", "ru", "ja", "de", "es", "fr"]
lang_grouped = df["language"].where(df["language"].isin(top_langs), other="other")
lang_dummies = pd.get_dummies(lang_grouped, prefix="lang", dtype=int)

# Encode Game Name
game_dummies = pd.get_dummies(df["game_name"], prefix="game", dtype=int)

# Build X
tag_and_label_cols = label_cols + ["label_none"] + list(SUPER_TAGS.keys())
X = pd.concat(
    [X, day_dummies, lang_dummies, game_dummies, df[tag_and_label_cols].fillna(0).astype(int)],
    axis=1
)

print(f"Final feature matrix X shape: {X.shape}")
print(X.head(5))

Final feature matrix X shape: (1992, 52)
   log_view_count  duration  is_featured  is_branded_content  day_Friday  day_Monday  \
0       12.282946      10.7            0                   0           0           0   
1       12.072393      15.5            0                   0           0           0   
2       11.918930       6.1            0                   0           0           0   
3       11.686139      24.6            0                   0           0           0   
4       11.523103       4.9            0                   0           0           0   

   day_Saturday  day_Sunday  day_Thursday  day_Tuesday  ...  label_DebatedSocialIssuesAndPolitics  \
0             0           0             0            1  ...                                     0   
1             0           1             0            0  ...                                     0   
2             0           0             0            1  ...                                     0   
3             1           

## Clustering

In [53]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# first we standardize the features
X_std = (X - X.mean()) / X.std()

# 2. Cluster for a few values of k and compare silhouette scores
for k in [4, 5, 6, 8]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_std)
    score = silhouette_score(X_std, labels)
    print(f"k = {k} | Silhouette Score: {score:.4f} | Cluster sizes: {np.bincount(labels)}")

k = 4 | Silhouette Score: 0.0681 | Cluster sizes: [708 103 327 854]
k = 5 | Silhouette Score: 0.0792 | Cluster sizes: [ 224   99 1228  105  336]
k = 6 | Silhouette Score: 0.0805 | Cluster sizes: [240 268 102 979  98 305]
k = 8 | Silhouette Score: 0.1004 | Cluster sizes: [ 98 199 336 126 198 662  98 275]
